In [1]:
# Decoding 

# Pool data across session and animals
import argparse
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.io as sio
import seaborn as sns
import pickle
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR, LinearSVC
from sklearn.metrics import (
    accuracy_score,
    silhouette_score,
    adjusted_rand_score,
    silhouette_samples,
    confusion_matrix,
)
from sklearn.cluster import AgglomerativeClustering, SpectralClustering, KMeans
from sklearn.model_selection import KFold, LeaveOneOut, train_test_split,  cross_val_score, cross_val_predict
from sklearn.model_selection import GridSearchCV
from sklearn.kernel_ridge import KernelRidge
from sklearn import linear_model
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from patsy import (
    ModelDesc,
    EvalEnvironment,
    Term,
    EvalFactor,
    LookupFactor,
    dmatrices,
    INTERCEPT,
)
from statsmodels.distributions.empirical_distribution import ECDF
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.colorbar as colorbar
import sys
import utils_py3 as ut

from s2p_utils.data_loader import DataLoader
from s2p_utils.processing_utils import (
    correct_overlapping_cells_across_planes,
    correct_timestamps,
    get_cell_only_activity,
    extract_events,
    get_corrected_F,
    normalize_signal,
    extract_Fave_around_events,
    extract_F_around_events,
    reorder_clusters,
    filter_trials_by_minITI
)
from plot_utils import (
    plot_raw_licks,
    plot_average_PSTH_around_interest_window,
    plot_individual_cells_activity,
    plot_PC_screenplot,
    plot_PCs,
    make_silhouette_plot,
    plot_activity_clusters,
    plot_cluster_pairs,
    plot_individual_trial_average_activity,
)


logger = logging.getLogger(__name__)

In [5]:
def extract_trials(animal_data, time_slice):
    """
    animal_data: np.array of shape (3, n_trials, n_cells, 20)
    time_slice: slice object (e.g., slice(0, 5) for cue period)
    
    Returns:
        X: (3 * n_trials, n_cells * len(time_slice))
        y: (3 * n_trials,)
    """
    n_CS, n_trials, n_cells, _ = animal_data.shape
    period_data = animal_data[..., time_slice]  # shape: (3, n_trials, n_cells, n_timepoints)
    reshaped = period_data.reshape(n_CS * n_trials, n_cells * period_data.shape[-1])
    labels = np.repeat(np.arange(n_CS), n_trials)  # 0, 1, 2 for CS1+, CS2+, CS3−
    return reshaped, labels


def get_decoded_animals(
    animal_list,
    patterns,
    trial_types,
    total_bins_length,
    decode_by_cluster=False,
    cluster_labels=None,
    selected_clusters=None,
    animal_id=None,
):
    """
    Prepares a dictionary of decoded animal data for decoding or PSTH plotting.

    Returns:
        decoded_animals (dict): keys are animal IDs, values contain data, local cell indices, and metadata.
    """
    if decode_by_cluster:
        animal_cell_starts = {}
        start_idx = 0
        for a in animal_list:
            n_cells = np.sum(np.char.find(animal_id.astype(str), a) >= 0)
            animal_cell_starts[a] = start_idx
            start_idx += n_cells
        
    decoded_animals = {}

    for animal in animal_list:
        data = patterns[animal]
        n_trial_per_cue = data.shape[0] // len(trial_types)
        total_cells = data.shape[1] // total_bins_length

        assert data.shape[1] % total_bins_length == 0, \
            f"Unexpected number of features in {animal}: {data.shape[1]}"

        if decode_by_cluster:
            # Get global indices of matching cells
            global_mask = (
                (np.char.find(animal_id.astype(str), animal) >= 0) &
                np.isin(cluster_labels, selected_clusters)
            )
            global_indices = np.where(global_mask)[0]
            local_start = animal_cell_starts[animal]
            local_indices = global_indices - local_start

            if len(local_indices) == 0:
                print(f"[{animal}] No cells in selected clusters {selected_clusters}. Skipping.")
                continue
        else:
            local_indices = np.arange(total_cells)

        decoded_animals[animal] = {
            "data": data,
            "local_cell_indices": local_indices,
            "n_trial_per_cue": n_trial_per_cue,
            "n_cells": len(local_indices),
        }

    return decoded_animals


def run_time_resolved_decoding(
    decoded_animals,
    animal_list,
    decoding_time_window,
    decoding_pair,
    total_bins_length,
    niteration,
    subsampling,
    clf,
    clf_chance,
    seed=0
):
    """
    Perform time-resolved decoding with leave-one-trial-out cross-validation.

    Args:
        decoded_animals (dict): Output from `get_decoded_animals`, per animal.
        animal_list (list): List of animals to decode.
        decoding_time_window (array): Time bins to decode at.
        decoding_pair (tuple): Trial types to decode (e.g., ("CS1", "CS3")).
        total_bins_length (int): Number of bins per cell.
        niteration (int): Number of iterations (for subsampling).
        subsampling (float or np.nan): Proportion of cells to subsample.
        clf (sklearn classifier): Main classifier.
        clf_chance (sklearn classifier): For chance decoding.
        seed (int): Random seed.

    Returns:
        accuracy (np.ndarray): Shape (n_animals, n_timepoints).
        accuracy_chance (np.ndarray): Same shape, for shuffled labels.
    """
    accuracy = [[] for _ in animal_list]
    accuracy_chance = [[] for _ in animal_list]

    for t in decoding_time_window:
        for ia, animal in enumerate(animal_list):

            d = decoded_animals[animal]
            data = d["data"]
            local_cell_indices = d["local_cell_indices"]
            n_cells = d["n_cells"]
            n_trial_per_cue = d["n_trial_per_cue"]

            time_indices = local_cell_indices * total_bins_length + t

            cue_map = {
                "CS1": data[0:n_trial_per_cue, time_indices],
                "CS2": data[n_trial_per_cue: 2 * n_trial_per_cue, time_indices],
                "CS3": data[2 * n_trial_per_cue: 3 * n_trial_per_cue, time_indices],
            }
            cs_a = cue_map[decoding_pair[0]]
            cs_b = cue_map[decoding_pair[1]]

            performance = []
            performance_chance = []

            for iiter in range(niteration):
                performance_temp = []
                performance_chance_temp = []

                if np.isnan(subsampling):
                    cell_idx = np.arange(n_cells)
                else:
                    n_sub = int(n_cells * subsampling)
                    cell_idx = np.random.choice(n_cells, n_sub, replace=False)

                for itrial in range(n_trial_per_cue):
                    cs_a_train = np.delete(cs_a, itrial, axis=0)[:, cell_idx]
                    cs_b_train = np.delete(cs_b, itrial, axis=0)[:, cell_idx]

                    traindata = np.vstack((cs_a_train, cs_b_train))
                    trainlabel = np.array([0] * (n_trial_per_cue - 1) + [1] * (n_trial_per_cue - 1))
                    testdata = np.vstack((cs_a[itrial, cell_idx], cs_b[itrial, cell_idx]))

                    clf.fit(traindata, trainlabel)
                    testlabel = clf.predict(testdata)
                    performance_temp.append(testlabel == [0, 1])

                    np.random.seed(seed)
                    shufflelabel = np.random.permutation(trainlabel)
                    clf_chance.fit(traindata, shufflelabel)
                    testlabel = clf_chance.predict(testdata)
                    performance_chance_temp.append(testlabel == [0, 1])

                performance.append(np.mean(np.concatenate(performance_temp)))
                performance_chance.append(np.mean(np.concatenate(performance_chance_temp)))

            accuracy[ia].append(np.mean(performance))
            accuracy_chance[ia].append(np.mean(performance_chance))

    return np.array(accuracy), np.array(accuracy_chance)


In [6]:
# 1. Initialize parameters
framerate = 5
trial_types = ["CS1+", "CS2+", "CS3-"]
pre_cue_window = 3
post_cue_window = 17
delay_to_reward = 3
# min_cell_prob = 0.5
# neucoeff = 0.7
# cell_threshold = 10
data_dir = "Z:\\2p\\experiment1"

imaging_system = "INSS"
learning_stage = "early"

# Set animals and days for early and late learning
if learning_stage == "early":
    result_dir = "Z:\\2p\\experiment1\\population_data\\early learning\\"
    animal_list = [
        "MZ_CA1_WD_F3",
        "MZ_CA1_WD_M4",
        "MZ_CA1_WD_M5",
        "MZ_CA1_WD_M6",
        "MZ_CA1_WD_M7",
        "MZ_CA1_WD_M8",
        "MZ_CA1_WD_JB_54",
        "MZ_CA1_WD_JB_55",
    ]
    daylist = [1, 1, 1, 2, 1, 1, 3, 3]

elif learning_stage == "late":
    result_dir = "Z:\\2p\\experiment1\\population_data\\late learning\\"
    animal_list = [
        "MZ_CA1_WD_F3",
        "MZ_CA1_WD_M4",
        "MZ_CA1_WD_M5",
        "MZ_CA1_WD_M6",
        "MZ_CA1_WD_M7",
        "MZ_CA1_WD_M8",
        "MZ_CA1_WD_JB_54",
        "MZ_CA1_WD_JB_55"
    ]   
    daylist = [7, 5, 6, 6, 5, 6, 8, 12]

In [31]:
file_dir = 'Z:\\2p\\experiment1\\MZ_CA1_WD_F3\\d1\\files'
F_5hz = np.load(os.path.join(file_dir, "F_5hz.npy"), allow_pickle=True)
F = np.load(os.path.join(file_dir, "F.npy"), allow_pickle=True)

In [40]:
len(F_5hz[0][1])

17970

In [7]:
def extract_Fave_around_events(
    CS,
    F,
    im_ts,
    num_planes: int,
    pre_cue_window: int,
    post_cue_window: int,
):
    """
    This function first generates Fcorrected traces around each cues based on input images indexes,
    and average Fcorr across all CS trials within the same CS type for each cell,
    and append each cell's average activity under each cue.

    Args:
        CS: all CS trials
        F: Fcorrected trace for all planes all cells
        im_dx: image indexes around each cue
        num_planes: number of planes

    Returns:
    Fcorrected_around_cue with the structure of len(CS), number of cells, timepoints

    """
    
    # Extract time around each cue and sorted by CS type, shape is numCS --> len trials
    interest_intervals = extract_interest_time_intervals(
        CS, pre_cue_window, post_cue_window
    )
    # Extract image time points around each cue and sorted by CS type and plane, shape is plane --> numCS --> len trials
    im_idx_around_cue = extract_imaging_ts_around_events(
        CS, im_ts, num_planes, interest_intervals
    )

    # F_ave_around_cues = [[] for _ in range(len(CS))]
    F_ave_around_cues_baseline_subtract = [[] for _ in range(len(CS))]

    framenumber = len(
        F[0][0][im_idx_around_cue[0][0][1]]
    )  # reference frame number equals the first cell's second trial from the first plane
    framespersecond = framenumber // (pre_cue_window + post_cue_window)

    for cue_type, cs in enumerate(CS):  # cue_type = 0,1,2 (CS1, CS2, CS3)
        for ip in range(num_planes):
            cue_ts = im_idx_around_cue[ip][
                cue_type
            ]  # image indexes for all trials in this cue type, holds same for all cells within the plane (trial number x framenumber)
            for cell in range(len(F[ip])):
                cell_F = []
                for trial in range(len(cs)):
                    F_temp = F[ip][cell][
                        cue_ts[trial]
                    ]  # F for cell in the plane, of this trial in this cue type (framenumber x )
                    # Correct for frame for each trial
                    if len(F_temp) > framenumber:
                        # if images number is bigger than default, drop the extra ones
                        F_temp = F_temp[0:framenumber]
                    elif len(F_temp) < framenumber:
                        # if images is smaller than default, add nan at the end to fill the spots
                        for i in range(framenumber - len(F_temp)):
                            F_temp = np.append(F_temp, np.nan)
                    cell_F.append(F_temp)
                # average across cs trials
                # cellave = np.nanmean(np.array(cell_F), axis=0)
                # baseline = np.nanmean(cellave[0 : pre_cue_window * framespersecond])
                # baselinesubtract = list(cellave - baseline)
                F_ave_around_cues_baseline_subtract[cue_type].append(cell_F)
                # F_ave_around_cues[cue_type].append(cellave)
    F_ave_around_cues_baseline_subtract = np.array(F_ave_around_cues_baseline_subtract)
    return F_ave_around_cues_baseline_subtract
